# 02 – Data understanding and data quality (Tasks 1–2)

OULAD as collected in Milestone 2 (seven CSV tables, SHA-256 pinned). This notebook reads the outputs of `src/01_understand.py` and `src/02_profile.py`; every number comes from `data/metadata/understanding.json` or `profile.json`.

In [1]:
import json, sys
from pathlib import Path
import polars as pl
from IPython.display import Markdown, display
PKG = Path.cwd().parent
sys.path.insert(0, str(PKG / "src"))
META, PROCESSED = PKG / "data" / "metadata", PKG / "data" / "processed"
pl.Config.set_tbl_rows(30); pl.Config.set_tbl_cols(20); pl.Config.set_fmt_str_lengths(80)
show = lambda name: display(Markdown((META / name).read_text()))

## Raw files are the Milestone 2 files

In [2]:
pl.read_csv(META / 'raw_check.csv')

file,bytes,sha256,pinned_sha256,data_rows_m2,check,read_only
str,i64,str,str,i64,str,str
"""courses.csv""",526,"""4f16eee7454b15e109b0a21a0e43be820e6846ed6f9301bb7feb5ab5ad737a75""","""4f16eee7454b15e109b0a21a0e43be820e6846ed6f9301bb7feb5ab5ad737a75""",22,"""MATCH""","""yes"""
"""assessments.csv""",8200,"""8cc738fb88ad760571d6f2a23059bfee0ffcae3bcd830514c9cbd5c6d5a046f1""","""8cc738fb88ad760571d6f2a23059bfee0ffcae3bcd830514c9cbd5c6d5a046f1""",206,"""MATCH""","""yes"""
"""vle.csv""",260126,"""d1b28303dea802ad87b4484e1196e878e06824850b9a4fe8aa34693439fe87e9""","""d1b28303dea802ad87b4484e1196e878e06824850b9a4fe8aa34693439fe87e9""",6364,"""MATCH""","""yes"""
"""studentInfo.csv""",3461652,"""7e6f3e474a5eee00639d2a414a6c7e928745823c2d2c2563ca1780145f99b0d6""","""7e6f3e474a5eee00639d2a414a6c7e928745823c2d2c2563ca1780145f99b0d6""",32593,"""MATCH""","""yes"""
"""studentRegistration.csv""",1109984,"""0d32676285372aaf2e7a80304e5b274b4fba24313e2ca4c04317225e1ec90170""","""0d32676285372aaf2e7a80304e5b274b4fba24313e2ca4c04317225e1ec90170""",32593,"""MATCH""","""yes"""
"""studentAssessment.csv""",5690310,"""fd5320786328d05af841ee7dd4b5871b9dada3b9fe9d6a3642b2f42635510a6e""","""fd5320786328d05af841ee7dd4b5871b9dada3b9fe9d6a3642b2f42635510a6e""",173912,"""MATCH""","""yes"""
"""studentVle.csv""",453836331,"""52668253d876c5becbcb72185977152700cecab2942aca807fecc3dd54b937f0""","""52668253d876c5becbcb72185977152700cecab2942aca807fecc3dd54b937f0""",10655280,"""MATCH""","""yes"""


## Dataset Understanding Map (Task 1)

In [3]:
show('understanding_map.md')

# Dataset Understanding Map (Task 1)

Generated 2026-11-06T15:24:01Z by `src/01_understand.py` from `data/raw/`. Every key and relationship below is a computed check; full results in `understanding.json`.

| File/Table | Rows | Unit of Analysis | Primary Key | Foreign Key | Main Variables | Time Field | Role in AQ |
| :---- | :---- | :---- | :---- | :---- | :---- | :---- | :---- |
| courses.csv | 22 | module run (module × presentation) | code_module, code_presentation (unique) | – | module_presentation_length | module_presentation_length (days) | Run length; run end for exam due days (AQ1–AQ3) |
| assessments.csv | 206 | assessment of a module run | id_assessment (unique) | code_module, code_presentation → courses | assessment_type, weight | date (due day) | Due days of tasks: missed and late tasks (AQ1, AQ2), task-based features and baseline rule (AQ3) |
| vle.csv | 6,364 | learning-platform resource (site) of a module run | id_site (unique) | code_module, code_presentation → courses | activity_type | week_from, week_to (planned weeks) | Resource type of clicks (AQ2 activity mix); planned weeks rarely filled |
| studentInfo.csv | 32,593 | enrolment (student × module run) | code_module, code_presentation, id_student (unique) | code_module, code_presentation → courses | final_result, gender, region, highest_education, imd_band, age_band, num_of_prev_attempts, studied_credits, disability | – (final_result observed at run end) | Outcome and target (AQ1–AQ3); background and audit attributes (AQ2, AQ3, AQ4 fairness) |
| studentRegistration.csv | 32,593 | enrolment (student × module run) | code_module, code_presentation, id_student (unique) | code_module, code_presentation, id_student → studentInfo (1–1) | date_registration, date_unregistration | date_registration, date_unregistration (days) | KPI denominator (registered on day 14), withdrawal day, population at each cut-off |
| studentAssessment.csv | 173,912 | submission (student × assessment) | id_assessment, id_student (unique) | id_assessment → assessments; id_student (+ run of the assessment) → studentInfo | score, is_banked | date_submitted (day) | Submission and lateness (AQ1, AQ2, AQ3) |
| studentVle.csv | 10,655,280 | logged interaction record (student × resource × day; several rows per day possible) | none documented (no key) | id_site → vle; code_module, code_presentation, id_student → studentInfo | sum_click | date (day of interaction) | Platform activity by day and week (AQ2, AQ3) |

## Relationships

| Relationship | Result |
| :---- | :---- |
| courses→assessments | orphan_rows=0, parents_without_children=0, children_per_parent_min=5, children_per_parent_max=14 |
| courses→vle | orphan_rows=0, parents_without_children=0, children_per_parent_min=106, children_per_parent_max=529 |
| courses→studentInfo | orphan_rows=0, parents_without_children=0, children_per_parent_min=365, children_per_parent_max=2498 |
| studentInfo↔studentRegistration | info_without_registration=0, registration_without_info=0 |
| assessments→studentAssessment | orphan_rows=0, parents_without_children=18, children_per_parent_min=278, children_per_parent_max=1917 |
| studentInfo→studentAssessment (enrolment) | orphan_rows=0, enrolments_with_submissions=25843 |
| vle→studentVle | orphan_sites=0, sites_never_used=96 |
| studentInfo→studentVle (enrolment) | orphan_enrolments=0, enrolments_with_activity=29228, enrolments_without_activity=3365 |


In [4]:
U = json.loads((META / 'understanding.json').read_text())
{k: U['population'][k] for k in ['runs', 'enrolments', 'students', 'students_with_several_enrolments', 'run_length_days']}

{'runs': 22,
 'enrolments': 32593,
 'students': 28785,
 'students_with_several_enrolments': 3538,
 'run_length_days': [234, 269]}

In [5]:
U['outcome']

{'values': [['Pass', 12361],
  ['Withdrawn', 10156],
  ['Fail', 7052],
  ['Distinction', 3024]],
 'withdrawn_without_unregistration_day': 93,
 'not_withdrawn_with_unregistration_day': [['Fail', 9]],
 'unregistration_on_or_before_day0': 3097,
 'unregistration_after_run_end': 1,
 'unregistration_before_registration': 0}

In [6]:
U['time']

{'day_zero': 'start of the module run; all day fields are offsets from it (negative = before the start)',
 'registration_day_range': [-322, 167],
 'registration_after_day14': 58,
 'registration_missing': 45,
 'assessment_due_range': [12, 261],
 'assessments_without_due_day': [['Exam', 11]],
 'due_after_run_end': 0,
 'submission_day_range': [-11, 608],
 'banked_submissions': 1909,
 'banked_submission_day_range': [-1, -1],
 'banked_submitted_after_due': 0,
 'interaction_day_range': [-25, 269],
 'interaction_rows_before_day0': 688988,
 'interaction_rows_after_run_end': 0,
 'vle_sites_with_planned_week': 1121,
 'vle_planned_week_max': 29}

## Profile (Task 2): the ten aspects of brief section 7

In [7]:
P = json.loads((META / 'profile.json').read_text())
list(P)

['generated',
 'shape',
 'missing',
 'duplicates',
 'validity',
 'distribution',
 'outliers',
 'outcome',
 'temporal',
 'joins',
 'representativeness']

In [8]:
P['duplicates']['chance_test_two_row_groups']

{'groups': 1311841,
 'observed_equal_share': 0.3149,
 'expected_by_site': 0.2399,
 'expected_by_activity_type': 0.2173,
 'observed_by_activity_type_top6': [['forumng', 0.328, 895551],
  ['oucontent', 0.3219, 279561],
  ['quiz', 0.1151, 71751],
  ['ouwiki', 0.0769, 21555],
  ['questionnaire', 0.06, 14694],
  ['oucollaborate', 0.5955, 11921]]}

In [9]:
P['missing']['imd_band_by_region_top3'], P['missing']['imd_band_success_missing_vs_present']

([['North Region', 731, 1823, 0.401],
  ['Ireland', 266, 1184, 0.2247],
  ['South Region', 48, 3092, 0.0155]],
 [[False, 0.4655, 31482], [True, 0.6571, 1111]])

In [10]:
P['outcome']['population_at_cutoff'], P['outcome']['kpi_completion_rate']

({'d13': {'enrolments': 28069,
   'non_completion': 12708,
   'rate': 0.4527,
   'withdrawn': 5682,
   'fail': 7026},
  'd27': {'enrolments': 27535,
   'non_completion': 12156,
   'rate': 0.4415,
   'withdrawn': 5119,
   'fail': 7037}},
 0.5474)

In [11]:
P['temporal']['tasks_due_by_cutoff']

{'d13': {'runs_with_task_due': 1,
  'runs_with_two_or_more_due': 0,
  'tasks_due_per_run_values': [[0, 21], [1, 1]]},
 'd27': {'runs_with_task_due': 16,
  'runs_with_two_or_more_due': 1,
  'tasks_due_per_run_values': [[0, 6], [1, 15], [2, 1]]}}

In [12]:
P['joins']

{'studentAssessment→assessments rows': [173912, 173912],
 '…→studentInfo rows': [173912, 173912],
 'studentVle→vle rows': [10655280, 10655280],
 'enrolment⋈aggregated clicks (inner) rows': [32593, 29228],
 'enrolments lost by an inner join with activity': 3365,
 'naive submissions×click-rows join rows (many-to-many)': 89347197,
 'rule': 'aggregate studentVle and studentAssessment to the enrolment (or enrolment-week) grain before joining them; use left joins from the enrolment table so enrolments without activity are kept'}

## Cross-check with the earlier OULAD exploration

In [13]:
pl.read_csv(META / 'profile_crosscheck.csv')

measure,exploration,milestone3,equal
str,i64,i64,bool
"""studentVle exact duplicate rows""",787170,787170,true
"""clicks raw""",39605099,39605099,true
"""clicks after collapsing duplicates""",38343063,38343063,true
"""repeated student-site-day groups""",1614505,1614505,true
"""rows in repeated groups""",3810465,3810465,true
"""enrolments without activity""",3365,3365,true
"""enrolments without submission""",6750,6750,true
"""late submissions""",49318,49318,true
"""submissions after run end""",85,85,true


## Data Quality Issue Log

In [14]:
show('quality_issue_log.md')

# Data Quality Issue Log (Task 2, brief section 7.1)

Every row rests on a computed result: `P.` = `data/metadata/profile.json` (written by `src/02_profile.py`),
`U.` = `data/metadata/understanding.json` (written by `src/01_understand.py`). Severity is judged against the
analytical questions (AQ1 descriptive, AQ2 diagnostic, AQ3 predictive, AQ4 prescriptive): High = it would change a
reported rate, the target or a feature at a cut-off; Medium = it would change a measure if ignored, with a known
size; Low = it affects a few records or fields that no AQ relies on. The decision column states what the pipeline
does; the Preparation Decision Matrix (`decision_matrix.md`) gives the rationale, implementation and validation.

| Issue ID | Variable / table | Evidence | Impact on AQ | Severity | Decision |
| :---- | :---- | :---- | :---- | :---- | :---- |
| DQ01 | studentVle: exact duplicate rows | 787,170 rows (7.39%) repeat another row in every column; 3.82–11.73% per run; 69.8% of duplicated groups are single-click; collapsing them removes 3.19% of clicks (P.duplicates). In two-row student-site-day groups, 31.5% have equal counts against 24.0% expected by chance from each site's own counts; quizzes, wikis and questionnaires fall below chance, forums and content pages above it (P.duplicates.chance_test_two_row_groups) | Click measures of AQ2 and AQ3 | Medium | Cause not decidable from the data: keep every row in the primary click measures (the table's record grain) and build a de-duplicated variant of each click feature for sensitivity |
| DQ02 | studentVle: grain differs from the documentation | 1,614,505 student-site-day groups hold several rows (3,810,465 rows), 1,179,074 of them with different click counts; up to 10 rows in one group (P.shape.studentVle_grain) | Any join or count at row level would mistake records for days | Medium | Aggregate clicks to enrolment × day and enrolment × week before any join; count active days, not rows |
| DQ03 | studentVle, studentAssessment: activity after unregistration | 29,440 click rows (98,745 clicks) by 1,279 enrolments, up to 423 days after unregistering; 588 non-banked submissions by 411 enrolments (P.validity) | Engagement up to withdrawal (AQ1, AQ2); none at a cut-off for students still registered | Medium | Truncate engagement measures at the unregistration day; feature views contain only students still registered at the cut-off, so no post-withdrawal event enters them |
| DQ04 | final_result × date_unregistration | 93 Withdrawn enrolments without an unregistration day, 87 of them in October 2014 runs; 9 Fail enrolments with one, 8 of them on or before day 0 (U.outcome, P.validity.withdrawn_without_day_by_presentation) | Populations at day 14, 13 and 27; time to withdrawal | Low | Keep final_result as the outcome; keep the recorded day; flag both groups (`withdrawal_day_missing`, `unregistered_not_withdrawn`); the 93 stay in every population because no record shows them leaving before a cut-off, and an evaluation flag lets test metrics be reported with and without them |
| DQ05 | studentRegistration: registration after day 14 | 58 enrolments registered on days 16–167, after the two-week sign-up window of the descriptor; 16 of them later unregistered (P.validity, check log T08) | KPI denominator and populations at cut-offs | Low | Define "enrolled at day d" as registered by day d (or registration day unknown) and not unregistered by day d; late registrants enter only the populations of later days |
| DQ06 | studentRegistration: departures before teaching | 3,097 enrolments unregistered on or before day 0; one unregistration on day 444 of a 268-day run (U.outcome) | KPI denominator | Low | Not an error: these students left before the run began and are outside the day-14 KPI population; the day-444 record is kept with a flag |
| DQ07 | studentAssessment: banked results | All 1,909 banked results carry date_submitted = −1, a placeholder for a result carried over from an earlier run (P.validity.banked_day_values) | Timeliness and missed-task measures (AQ1–AQ3) | Medium | Exclude banked rows from submission timing, lateness and missed-task counts; keep a per-enrolment count of banked results, known at registration |
| DQ08 | assessments, studentAssessment: exams | 18 of 24 exams have no result rows (scores only for CCC 2 and DDD 4); 11 exams have no due day (P.missing, U.time) | "First missed assessed task" (AQ1); task signs (AQ2, AQ3) | High for exams, none for coursework | Count missed and late tasks over TMA and CMA only; the outcome stays final_result; exam scores are not used |
| DQ09 | assessments: weights | Coursework weights sum to 0 in the three GGG runs and to 100 elsewhere; FFF CMAs weigh 0; CCC 2014 runs hold two exams (total 200) (P.validity, U.measurement) | Weighted score features | Medium | Build no weighted grade; count tasks unweighted (formative tasks are still due) and keep weight as a column |
| DQ10 | studentAssessment: late and post-run submissions | 49,318 of 169,138 non-banked submissions with a due day are late (29.2%; median 3, 99th percentile 29 days): 16,320 of 97,088 TMA and 32,991 of 69,956 CMA, although the OU's current (2025) policy accepts no CMA after its cut-off day; 85 fall after the run ended, up to day 608 (P.validity) | Late-task signs (AQ1, AQ2, early-sign rule) | Medium | Keep every submission; lateness = date_submitted − due day, counted separately for TMA and CMA because the meaning of a late CMA in 2013–2014 is not documented; post-run submissions fall after both cut-offs |
| DQ11 | studentAssessment.score | 173 missing scores, all TMA (P.missing.score_missing_by_type); 45.8% of CMA scores are multiples of 10 against 17.4% of TMA (P.distribution) | Score features | Low | Keep the submission, leave the score missing; scores enter features only where the mark is known at the cut-off (decision matrix, row "Derived feature") |
| DQ12 | studentInfo.imd_band | 1,111 missing (3.41%), 40.1% of North Region and 22.5% of Ireland; success 65.7% when missing against 46.6% when present, so not missing at random; one label "10-20" lacks the % sign (P.missing, P.validity) | AQ4 fairness check by deprivation; background features | Medium | Normalise the label to "10-20%"; keep missing as its own category "Missing"; no imputation |
| DQ13 | studentRegistration.date_registration | 45 missing, 39 of them Withdrawn (P.missing) | Registration-lead feature | Low | Keep null with an indicator; any filling is fitted on the training runs only |
| DQ14 | vle.week_from, week_to | 82.4% missing; filled for 1,121 of 6,364 sites (P.missing, U.time) | None | Low | Not used |
| DQ15 | studentVle: days without any click | In all six February 2014 runs two consecutive days carry no click (days 218–219; days 211–212 in BBB) (P.temporal) | Daily activity series after week 30 | Low | Treat the two days as missing, not as zero activity, in daily series; both lie after the cut-offs |
| DQ16 | Outliers: clicks and credits | One row of 6,977 clicks; clicks per enrolment skewness 3.03, 2,503 enrolments beyond the upper interquartile-range fence; studied_credits up to 655 against a 99th percentile of 240 (P.distribution) | Scale-sensitive methods | Low | Keep every value (heavy users and students on several modules are real); log or rank transforms are applied in modelling, fitted on training runs |
| DQ17 | Enrolments without any activity or submission row | 3,365 enrolments without a click row (2,988 Withdrawn; 2,390 left by day 0); 6,750 without a submission (P.missing) | Counts would drop these students under an inner join | Medium | Absence of a row records no activity (descriptor: no submission, no result row): fill counts with 0 after left joins from the enrolment table |
| DQ18 | assessments: tasks due by the cut-offs | A non-exam task is due by day 13 in 1 of 22 runs and by day 27 in 16; two or more are due by day 27 in 1 run only (P.temporal.tasks_due_by_cutoff) | AQ3 comparison rule and secondary KPI 1 ("two or more tasks missed or late by week 4") can fire in 1 run of 22 | High | Task 3 decision (AQ–Data fit): rule restated on the tasks actually due (Change Log C10) |
| DQ19 | Population and module mix | Seven UK distance-learning modules of 2013–2014, selected partly for many failing students; women 11.5% of EEE and 88.4% of BBB; completion in the day-14 population from 45.7% (CCC) to 73.8% (AAA) (P.representativeness) | Generalisation of every AQ; group comparisons | High for claims | State the scope; compare groups within module; results describe OULAD and are re-checked on the university's data before decisions D1–D3 |
